# Tree-Based Models

Unlike the previous notebook, this stage of the project is devided into two separate notebooks based on the preprocessing pipeline required by each model. RandomForest and XGBoost, both rely on the same preprocessing approach and are therefore evaluated together in this notebook. CatBoost, however, uses a different preprocessing strategy, by handling categorical features natively and is evaluated separately in the next notebook.

Raandom Forest serves as the baseline tree-based model for both notebooks. Its evaluated metrics are saved to a CSV file so they can be reused in the CatBoost notebook, allowing all tree-based models to be compared consistently without retraining the baseline model.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

df_copy = builder.get_df(df)

df_copy.shape

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(74111, 27)

# Baseline Model. RandomForest

In [2]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 25), (14823, 25))

In [4]:
import preprocessing.tree_preprocessor as preprocessor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

cat_features = X_train.select_dtypes(include=['string', 'object']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

tree_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

tree_pipeline.fit(X_train, y_train)

y_pred_test_log = tree_pipeline.predict(X_test)
y_pred_train_log = tree_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 52.22$ | Train MAE: 21.08$
Test RMSE: 118.45$ | Train RMSE: 56.19$
Test R2 Score: 0.68 | Train R2 Score: 0.95


## RandomForest Conclusion

The baseline RandomForest model exhibit significant overfitting, with substantially better performance on the training set that on the test set. Therefore, its current evaluation metrics are not suitable as the primary baseline for comparing tree-based models. In the next step, hyperparameter tuning will be performed using GridSearchCV to reduce overfitting and establish a more reliable baseline for subsequent comparisons. 

# RandomForest. Hyperparameter Tuning.

In [9]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

BASELINE_RANDOM_SEARCH_MODEL = ARTIFACTS_DIR / "random_forest_random_search.joblib"

In [12]:
%%time

import joblib
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

if BASELINE_RANDOM_SEARCH_MODEL.exists():
    print("Loading RandomizedSearchCV...")
    random_search = joblib.load(BASELINE_RANDOM_SEARCH_MODEL)
else:
    print("Training RandomizedSearchCV...")
    
    tree_pipeline = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ])
    
    param_dist = {
        "model__n_estimators": randint(100, 600),
        "model__max_depth": [10, 20, 30, 40, None],
        "model__min_samples_split": randint(2, 20),
        "model__min_samples_leaf": randint(1, 10),
        "model__max_features": ['log2', 'sqrt', 0.5]
    }
    
    random_search = RandomizedSearchCV(
        estimator=tree_pipeline,
        param_distributions=param_dist,
        n_iter=15,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=2
    )

    random_search.fit(X_train, y_train)
    joblib.dump(random_search, BASELINE_RANDOM_SEARCH_MODEL)

best_rf = random_search.best_estimator_
random_search.best_params_

Loading RandomizedSearchCV...
CPU times: user 82.5 ms, sys: 67.1 ms, total: 150 ms
Wall time: 149 ms


{'model__max_depth': 40,
 'model__max_features': 0.5,
 'model__min_samples_leaf': 3,
 'model__min_samples_split': 13,
 'model__n_estimators': 154}

In [13]:
y_pred_test_log = best_rf.predict(X_test)
y_pred_train_log = best_rf.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 52.00$ | Train MAE: 37.49$
Test RMSE: 118.74$ | Train RMSE: 92.21$
Test R2 Score: 0.69 | Train R2 Score: 0.84
